# Final figures and interpretation

This notebook generates the three final benchmark figures and writes two biological
interpretation reports from available output tables.

The figures and reports are derived solely from CSVs produced by earlier pipeline stages.
No AnnData file is loaded here — all inputs are pre-computed tables.

Mirrors:
- `scripts/generate_final_figures.py`
- `scripts/write_interpretation_reports.py`

In [ ]:
import subprocess
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
for _p in [PROJECT_ROOT, PROJECT_ROOT / "src"]:
    if str(_p) not in sys.path:
        sys.path.insert(0, str(_p))
assert (PROJECT_ROOT / "src" / "rarecell").exists(), PROJECT_ROOT
PROJECT_ROOT

## Required inputs

All inputs are pre-computed CSVs.  Missing files are tolerated — the scripts
substitute placeholder panels rather than failing.

| Input | Produced by | Description |
|---|---|---|
| `results/metrics/multi_target_benchmark_raw.csv` | `multi_target_benchmark.ipynb` | Primary raw benchmark (preferred over single-target raw) |
| `results/metrics/rare_cell_benchmark_raw.csv` | `rare_cell_downsampling_benchmark.ipynb` | Fallback raw benchmark if multi-target CSV is absent |
| `results/tables/cross_target_method_ranking.csv` | `multi_target_benchmark.ipynb` | Cross-target representation ranking |
| `results/tables/biological_interpretation_summary.csv` | `error_analysis.ipynb` | Marker + failure-mode context |
| `results/tables/target_neighbor_composition.csv` | `error_analysis.ipynb` | Neighbor label composition |
| `results/tables/target_cell_error_analysis.csv` | `error_analysis.ipynb` | Target-cell error absorption |
| `results/tables/marker_gene_summary.csv` | `marker_analysis.ipynb` | RNA marker table |
| `results/tables/marker_protein_summary.csv` | `marker_analysis.ipynb` | Protein marker table |
| `config/multi_target_config.yaml` | Manual | Dataset path and label column for report metadata |

## Canonical script commands

```bash
python scripts/generate_final_figures.py \
    --config config/multi_target_config.yaml \
    --results-dir results \
    --output-dir results/figures

python scripts/write_interpretation_reports.py --config config/multi_target_config.yaml
```

Key functions used by `generate_final_figures.py`:
- `rarecell.plotting.plot_study_design(output_path, config)` — Figure 1: study-design schematic
- `rarecell.plotting.plot_final_figure_2_main_benchmark(raw, output_path)` — Figure 2: main F1 benchmark
- `rarecell.plotting.plot_final_figure_3_failure_modes(biological, neighbors, errors, output_path)` — Figure 3
- `rarecell.plotting.plot_multi_target_metric_curve(raw, metric, output_path)` — per-metric curves
- `rarecell.plotting.plot_cross_target_method_ranking(ranking, output_path)` — ranking bar chart

Key functions used by `write_interpretation_reports.py`:
- Reads `benchmark_summary.csv` (or `multi_target_benchmark_summary.csv`), marker tables, and
  biological interpretation table, then writes two structured Markdown reports.

In [ ]:
for cmd in [
    [
        sys.executable,
        "scripts/generate_final_figures.py",
        "--config", "config/multi_target_config.yaml",
        "--results-dir", "results",
        "--output-dir", "results/figures",
    ],
    [
        sys.executable,
        "scripts/write_interpretation_reports.py",
        "--config", "config/multi_target_config.yaml",
    ],
]:
    result = subprocess.run(cmd, cwd=PROJECT_ROOT, text=True, capture_output=True)
    print(result.stdout)
    if result.returncode != 0:
        print(result.stderr)
        raise SystemExit(result.returncode)

## Expected outputs

| File | Script | Description |
|---|---|---|
| `results/figures/final_figure_1_study_design.png` | `generate_final_figures.py` | Study-design schematic |
| `results/figures/final_figure_2_main_benchmark.png` | `generate_final_figures.py` | Main F1-vs-fraction benchmark figure |
| `results/figures/final_figure_3_failure_modes.png` | `generate_final_figures.py` | Failure-mode interpretation figure |
| `results/figures/multi_target_f1_curve.png` | `generate_final_figures.py` | F1 curves for all targets |
| `results/figures/multi_target_neighborhood_purity_curve.png` | `generate_final_figures.py` | Neighborhood purity curves |
| `results/figures/cross_target_method_ranking.png` | `generate_final_figures.py` | Cross-target ranking bar chart |
| `results/tables/final_results_table.csv` | `write_interpretation_reports.py` | Compact joined table for manuscript reporting |
| `results/reports/biological_interpretation.md` | `write_interpretation_reports.py` | Marker availability, failure modes, conservative interpretation |
| `results/reports/final_results_interpretation.md` | `write_interpretation_reports.py` | Main result, cross-target comparison, limitations, candidate abstract |

In [ ]:
outputs = [
    "results/figures/final_figure_1_study_design.png",
    "results/figures/final_figure_2_main_benchmark.png",
    "results/figures/final_figure_3_failure_modes.png",
    "results/tables/final_results_table.csv",
    "results/reports/biological_interpretation.md",
    "results/reports/final_results_interpretation.md",
]
[(path, (PROJECT_ROOT / path).exists()) for path in outputs]

## Summary

In [ ]:
print("Generated outputs:")
for path in outputs:
    p = PROJECT_ROOT / path
    status = "OK" if p.exists() else "MISSING"
    print(f"  [{status}] {path}")
print()
print("Deviations from scripts:")
print("  - generate_final_figures.py skips figures gracefully when source tables are")
print("    absent; placeholder panels are substituted rather than raising errors.")
print("  - write_interpretation_reports.py fills TODO placeholders for missing tables.")
print("  - Run earlier pipeline stages (controls, marker_analysis, error_analysis,")
print("    multi_target_benchmark) to populate the source tables for complete figures.")